# Bài 49: Học tự giám sát và Học tương phản (Self-Supervised and Contrastive Learning)

Học chuyển giao ở Bài 38 vẫn cần một nhiệm vụ nguồn có nhãn để tiền huấn luyện một mạng xương sống hữu ích. **Học tự giám sát (self-supervised learning)** loại bỏ cả yêu cầu đó: tiền huấn luyện trên các ảnh *không nhãn* bằng cách tự tạo ra một nhiệm vụ từ chính dữ liệu, hoàn toàn không dùng chú thích nào của con người. Bài này xây dựng công thức thống trị cho điều đó, **học tương phản (contrastive learning)** (kiểu SimCLR, <a href="../references.html#chen-2020-simclr">Chen et al., 2020</a>): hai góc nhìn được tăng cường ngẫu nhiên của cùng một ảnh nên tạo ra các vector nhúng (embedding) tương tự; các góc nhìn của các ảnh khác nhau thì không.

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Hai góc nhìn, một danh tính (Two views, one identity)
Lấy 500 ảnh hình dạng không nhãn (các nhãn tồn tại trong tập dữ liệu tổng hợp này chỉ để có thể đo độ chính xác sau này — bước tiền huấn luyện bên dưới không bao giờ nhìn vào chúng). Với mỗi ảnh được lấy mẫu trong khi tiền huấn luyện, tạo hai góc nhìn được tăng cường độc lập (các phép lật và xoay ở Bài 8/34, cộng với nhiễu điểm ảnh).

In [ ]:
def make_image(shape_type, cx, cy, size=16):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        img[cy-1:cy+2, cx-3:cx+4] = 1.0
        img[cy-3:cy+4, cx-1:cx+2] = 1.0
    else:
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx-cx)**2 + (yy-cy)**2) <= 9] = 1.0
    return img

def make_dataset(rng_local, n, position_range=(4, 12)):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        imgs.append(make_image(shape_type, cx, cy))
        labels.append(0 if shape_type == 'plus' else 1)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.int64)

def augment(img, rng_local):
    if rng_local.random() < 0.5:
        img = np.fliplr(img).copy()
    angle = rng_local.uniform(-20, 20)
    M = cv2.getRotationMatrix2D((8, 8), angle, 1.0)
    img = cv2.warpAffine(img, M, (16, 16))
    return np.clip(img + rng_local.normal(0, 0.1, img.shape), 0, 1).astype(np.float32)

rng = np.random.default_rng(5)
X_unlabeled, y_unlabeled = make_dataset(rng, 500)

aug_rng = np.random.default_rng(1)
fig, axes = plt.subplots(2, 4, figsize=(9, 4.5))
for i in range(4):
    axes[0, i].imshow(augment(X_unlabeled[i], aug_rng), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(augment(X_unlabeled[i], aug_rng), cmap='gray'); axes[1, i].axis('off')
axes[0, 0].set_title('view 1', fontsize=9, loc='left')
axes[1, 0].set_title('view 2', fontsize=9, loc='left')
plt.suptitle('Two augmented views of the same 4 images')
plt.show()

## Hàm mất mát tương phản (NT-XEnt) (The contrastive loss (NT-XEnt))
Với một lô `N` ảnh, tạo ra `2N` vector nhúng (mỗi ảnh hai góc nhìn). Với mỗi vector nhúng, cặp dương thật sự duy nhất của nó là góc nhìn *còn lại* của cùng ảnh; mọi vector khác trong `2N-2` vector nhúng là một mẫu âm. Hãy coi đây là một bài toán phân loại — "vector nào trong 2N-1 vector nhúng còn lại là cặp dương của tôi?" — và cực tiểu hóa entropy chéo trên các độ tương đồng cosin. Đây là hàm mất mát **entropy chéo được chuẩn hóa và co giãn theo nhiệt độ (normalized temperature-scaled cross-entropy — NT-Xent)**.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, out_dim=16):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(), nn.AdaptiveMaxPool2d(1),
        )
        self.proj = nn.Linear(32, out_dim)

    def forward(self, x):
        return self.proj(self.conv(x).flatten(1))

def nt_xent_loss(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    z = torch.cat([z1, z2], dim=0)          # (2N, D)
    sim = z @ z.T / temperature              # (2N, 2N) cosine similarities
    N = z1.shape[0]
    mask = torch.eye(2 * N, dtype=torch.bool)
    sim.masked_fill_(mask, float('-inf'))    # exclude comparing an embedding to itself
    targets = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)])  # each row's true positive index
    return F.cross_entropy(sim, targets)

def train_contrastive(seed, epochs=200, lr=0.01, batch_size=64):
    torch.manual_seed(seed)
    encoder = Encoder()
    opt = torch.optim.Adam(encoder.parameters(), lr=lr)
    local_aug_rng = np.random.default_rng(seed + 100)
    n = len(X_unlabeled)
    for epoch in range(epochs):
        idx = np.random.default_rng(epoch).permutation(n)[:batch_size]
        batch = X_unlabeled[idx]
        view1 = np.stack([augment(im, local_aug_rng) for im in batch])
        view2 = np.stack([augment(im, local_aug_rng) for im in batch])
        z1 = encoder(torch.tensor(view1).unsqueeze(1))
        z2 = encoder(torch.tensor(view2).unsqueeze(1))
        loss = nt_xent_loss(z1, z2)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return encoder, loss.item()

encoder, final_loss = train_contrastive(seed=0)
print(f'final NT-Xent loss after 200 steps: {final_loss:.3f}  (random-chance loss would be ln(2*64-1) = {np.log(2*64-1):.3f})')

## Đánh giá bằng một đầu dò tuyến tính (Evaluating with a linear probe)
Cách tiêu chuẩn để kiểm tra xem việc tiền huấn luyện tự giám sát có thực sự học được thứ gì hữu ích không: đóng băng bộ mã hóa đã tiền huấn luyện (mẫu mạng xương sống đóng băng ở Bài 38) và chỉ huấn luyện một bộ phân loại tuyến tính nhỏ bên trên các đặc trưng của nó — một kỹ thuật gọi là **dò tuyến tính (linear probing)** — dùng một số lượng *nhỏ xíu* mẫu có nhãn, chính xác là chế độ ít nhãn mà tiền huấn luyện tự giám sát nhằm hỗ trợ. So sánh với cùng đầu dò tuyến tính đó trên một bộ mã hóa khởi tạo ngẫu nhiên (chưa bao giờ được huấn luyện).

In [ ]:
def make_noisy_dataset(rng_local, n, noise=0.35, position_range=(4, 12)):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        img = make_image(shape_type, cx, cy)
        img = np.clip(img + rng_local.normal(0, noise, img.shape), 0, 1).astype(np.float32)
        imgs.append(img)
        labels.append(0 if shape_type == 'plus' else 1)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.int64)

X_probe_train, y_probe_train = make_noisy_dataset(rng, 8)    # only 8 labeled examples
X_probe_test, y_probe_test = make_noisy_dataset(rng, 150)

def linear_probe_acc(enc, seed):
    torch.manual_seed(seed)
    with torch.no_grad():
        feat_train = enc(torch.tensor(X_probe_train).unsqueeze(1))
        feat_test = enc(torch.tensor(X_probe_test).unsqueeze(1))
    probe = nn.Linear(feat_train.shape[1], 2)
    opt = torch.optim.Adam(probe.parameters(), lr=0.05)
    ytr = torch.tensor(y_probe_train)
    for _ in range(300):
        opt.zero_grad()
        loss = F.cross_entropy(probe(feat_train), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        preds = probe(feat_test).argmax(1).numpy()
    return (preds == y_probe_test).mean()

random_accs, contrastive_accs = [], []
for seed in range(5):
    enc_contrastive, _ = train_contrastive(seed=seed)
    contrastive_accs.append(linear_probe_acc(enc_contrastive, seed=seed + 50))

    torch.manual_seed(seed)
    enc_random = Encoder()
    random_accs.append(linear_probe_acc(enc_random, seed=seed + 50))

print(f'linear probe on RANDOM (untrained) features:     {np.mean(random_accs):.1%}  (+/- {np.std(random_accs):.1%})')
print(f'linear probe on CONTRASTIVE-pretrained features: {np.mean(contrastive_accs):.1%}  (+/- {np.std(contrastive_accs):.1%})')

Chỉ với 8 mẫu có nhãn để huấn luyện đầu dò, bộ mã hóa được tiền huấn luyện tương phản cho một cải thiện thực sự (dù nhiễu, do có quá ít nhãn) so với các đặc trưng ngẫu nhiên — mục tiêu tương phản, dù chưa bao giờ thấy một nhãn nào, đã đẩy các ảnh cùng hình dạng về các điểm gần nhau trong không gian nhúng và các ảnh khác hình dạng ra xa nhau, hoàn toàn bằng cách yêu cầu các góc nhìn tăng cường của cùng một ảnh phải đồng thuận.

Đây là cơ chế đằng sau lý do tiền huấn luyện tự giám sát quan trọng ở quy mô lớn: các mô hình như DINOv2 (Bài 50) được tiền huấn luyện theo cách này trên hàng trăm triệu ảnh *không nhãn* — không thể gán nhãn thủ công ở quy mô đó — và các đặc trưng thu được chuyển giao sang các nhiệm vụ hạ nguồn chỉ với một nhúm mẫu có nhãn, đúng như đã minh họa ở đây ở dạng thu nhỏ.

### Bài tập (Exercises)
1. Tăng `out_dim` trong `Encoder` từ 16 lên 64. Một không gian nhúng nhiều chiều hơn có cải thiện độ chính xác của đầu dò tuyến tính, gây hại cho nó, hay tạo ra ít khác biệt ở quy mô dữ liệu này?
2. Thử `temperature=0.1` và `temperature=2.0` trong `nt_xent_loss` thay vì `0.5`. Nhiệt độ điều khiển mức độ gắt gao mà hàm mất mát phạt các mẫu âm suýt trúng — có thái cực nào làm thay đổi đáng kể độ chính xác đầu dò cuối cùng không?
3. Làm yếu các phép tăng cường trong `augment` (ví dụ khoảng xoay `(-5, 5)` thay vì `(-20, 20)`, không có nhiễu). Tiền huấn luyện tương phản có còn vượt đường cơ sở ngẫu nhiên không — và câu trả lời gợi ý gì về *lý do* cường độ tăng cường là một trong những siêu tham số được tinh chỉnh nhiều nhất trong các quy trình học tương phản thực tế?